In [1]:
import pandas as pd
import numpy as np
import ollama
import ipywidgets as widgets

from pptx import Presentation
from pptx.util import Inches, Pt
from IPython.display import display, FileLink
from pathlib import Path
from io import BytesIO

print("All required libraries imported successfully!")

All required libraries imported successfully!


In [2]:
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)

file_path = filedialog.askopenfilename(
    title="Select CSV Dataset",
    filetypes=[("CSV Files", "*.csv")]
)

root.destroy()

if file_path:
    df = pd.read_csv(file_path)
    print("CSV uploaded successfully!")
    print("Dataset shape:", df.shape)
    display(df.head())
else:
    print("No file selected.")

NameError: name 'pd' is not defined

In [3]:
import pandas as pd

df = pd.read_csv(file_path)

print("CSV uploaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

CSV uploaded successfully!
Dataset shape: (835, 6)


,Job Title,Location,Salary,Monthly Salary,Locality,State
0,Robotics / STEM Trainer,"Indore, Madhya Pradesh","₹1,80,000 - ₹3,60,000 a year",270000.0,NaN,Madhya Pradesh
1,HTML Developers - Freshers,"Banaswadi, Bengaluru, Karnataka",Not specified,NaN,Banaswadi,Karnataka
2,Java Developers - Freshers,"Banaswadi, Bengaluru, Karnataka",Not specified,NaN,Banaswadi,Karnataka
3,"Teachers - Pre-Primary and Primary (English, M...","Pulivendla, Andhra Pradesh","₹20,000 - ₹30,000 a month",25000.0,NaN,Andhra Pradesh
4,College Student,Remote,"₹15,000 - ₹20,000 a month",17500.0,Remote,Remote


In [4]:
print("===== DATASET OVERVIEW =====")
print("Total records:", len(df))
print("Total columns:", len(df.columns))

print("\n===== MISSING VALUES =====")
print(df.isnull().sum())

print("\n===== DUPLICATE RECORDS =====")
print(df.duplicated().sum())

print("\n===== TOP 5 STATES / REMOTE =====")
print(df["State"].value_counts().head(5))

print("\n===== TOP 5 JOB TITLES =====")
print(df["Job Title"].value_counts().head(5))

print("\n===== SALARY STATISTICS =====")
print(df["Monthly Salary"].describe())

===== DATASET OVERVIEW =====
Total records: 835
Total columns: 6

===== MISSING VALUES =====
Job Title           0
Location            0
Salary              0
Monthly Salary    338
Locality          581
State               0
dtype: int64

===== DUPLICATE RECORDS =====
5

===== TOP 5 STATES / REMOTE =====
State
Kerala         161
Tamil Nadu     148
Karnataka      106
Remote          89
Maharashtra     78
Name: count, dtype: int64

===== TOP 5 JOB TITLES =====
Job Title
Delivery Partner/Delivery Executive    9
Video Editor                           8
Trainee                                8
Business Development Associate         8
Management Trainee                     6
Name: count, dtype: int64

===== SALARY STATISTICS =====
count       497.000000
mean      63087.595050
std      126922.462764
min         180.000000
25%       14000.000000
50%       19250.000000
75%       28000.000000
max      939847.175000
Name: Monthly Salary, dtype: float64


In [6]:
import pandas as pd
import numpy as np
df_clean = df.drop_duplicates().copy()

df_clean["Locality"] = df_clean["Locality"].fillna("Not specified")

salary_text = df_clean["Salary"].astype(str).str.lower()

annual_mask = salary_text.str.contains(r"\byear\b|\bannual", regex=True)
monthly_mask = salary_text.str.contains(r"\bmonth\b", regex=True)

df_clean["Corrected Monthly Salary"] = pd.to_numeric(
    df_clean["Monthly Salary"], errors="coerce"
)

df_clean.loc[annual_mask, "Corrected Monthly Salary"] /= 12

df_clean.loc[
    ~(annual_mask | monthly_mask),
    "Corrected Monthly Salary"
] = np.nan

print("===== CLEANED DATASET =====")
print("Original records:", len(df))
print("Cleaned records:", len(df_clean))
print("Duplicates remaining:", df_clean.duplicated().sum())
print("Annual salary records:", annual_mask.sum())

print("\n===== CORRECTED SALARY STATISTICS =====")
print(df_clean["Corrected Monthly Salary"].describe())

display(df_clean.head())

===== CLEANED DATASET =====
Original records: 835
Cleaned records: 830
Duplicates remaining: 0
Annual salary records: 74

===== CORRECTED SALARY STATISTICS =====
count       492.000000
mean      22850.855302
std       23612.325253
min        3000.000000
25%       14000.000000
50%       18500.000000
75%       25000.000000
max      300000.000000
Name: Corrected Monthly Salary, dtype: float64


,Job Title,Location,Salary,Monthly Salary,Locality,State,Corrected Monthly Salary
0,Robotics / STEM Trainer,"Indore, Madhya Pradesh","₹1,80,000 - ₹3,60,000 a year",270000.0,Not specified,Madhya Pradesh,22500.0
1,HTML Developers - Freshers,"Banaswadi, Bengaluru, Karnataka",Not specified,NaN,Banaswadi,Karnataka,NaN
2,Java Developers - Freshers,"Banaswadi, Bengaluru, Karnataka",Not specified,NaN,Banaswadi,Karnataka,NaN
3,"Teachers - Pre-Primary and Primary (English, M...","Pulivendla, Andhra Pradesh","₹20,000 - ₹30,000 a month",25000.0,Not specified,Andhra Pradesh,25000.0
4,College Student,Remote,"₹15,000 - ₹20,000 a month",17500.0,Remote,Remote,17500.0


In [7]:
state_counts = df_clean["State"].value_counts()
job_counts = df_clean["Job Title"].value_counts()

salary = df_clean["Corrected Monthly Salary"]

remote_count = (df_clean["State"] == "Remote").sum()
remote_pct = remote_count / len(df_clean) * 100

missing_salary_pct = salary.isna().mean() * 100

insights = [
    f"Geographic Demand: {state_counts.idxmax()} leads "
    f"with {state_counts.max()} job listings.",

    f"Job Demand: {job_counts.idxmax()} is the most "
    f"frequently listed job title with {job_counts.max()} openings.",

    f"Salary Benchmark: The average monthly salary is "
    f"INR {salary.mean():,.2f}, while the median is "
    f"INR {salary.median():,.2f}.",

    f"Remote Opportunities: {remote_count} remote listings "
    f"represent {remote_pct:.2f}% of the dataset.",

    f"Salary Transparency: {missing_salary_pct:.2f}% of "
    f"listings lack usable monthly salary information."
]

print("===== 5 KEY BUSINESS INSIGHTS =====\n")

for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}\n")

===== 5 KEY BUSINESS INSIGHTS =====

1. Geographic Demand: Kerala leads with 159 job listings.

2. Job Demand: Delivery Partner/Delivery Executive is the most frequently listed job title with 9 openings.

3. Salary Benchmark: The average monthly salary is INR 22,850.86, while the median is INR 18,500.00.

4. Remote Opportunities: 89 remote listings represent 10.72% of the dataset.

5. Salary Transparency: 40.72% of listings lack usable monthly salary information.



In [8]:
import ollama

prompt = f"""
You are a senior business data analyst.

Dataset: Indian Job Market
Total job listings: {len(df_clean)}

Key analytical insights:
{chr(10).join(insights)}

Write a professional executive summary of approximately
200 words for business stakeholders.

Requirements:
- Explain the key findings.
- Highlight business opportunities and risks.
- Include relevant statistics.
- Use professional and clear business language.
- Do not invent additional statistics.
- Limit conclusions to the provided dataset.
"""

response = ollama.chat(
    model="qwen2.5:3b",
    messages=[
        {"role": "user", "content": prompt}
    ],
    options={"temperature": 0.2}
)

executive_summary = response["message"]["content"]

print("===== AI EXECUTIVE SUMMARY =====\n")
print(executive_summary)

print("\nWord count:", len(executive_summary.split()))

===== AI EXECUTIVE SUMMARY =====

Executive Summary

For business stakeholders, this summary outlines key insights from the Indian Job Market dataset, providing a comprehensive overview of current trends and opportunities. The analysis reveals that Kerala leads with 159 job listings, indicating strong demand in this region. The most frequently listed job title is Delivery Partner/Delivery Executive with 9 openings, suggesting a growing need for delivery services. The average monthly salary is INR 22,850.86, with a median of INR 18,500.00, highlighting the competitive nature of the job market. Remote opportunities represent 10.72% of the dataset, representing a significant portion of the market. However, 40.72% of listings lack usable monthly salary information, presenting a challenge for salary benchmarking and negotiation.

These insights underscore the importance of understanding regional market dynamics and the evolving job market landscape. For businesses, the opportunity lies in l

In [9]:
recommendation_prompt = f"""
You are a senior business strategy consultant.

Dataset: Indian Job Market

Key insights:
{chr(10).join(insights)}

Generate 5 actionable business recommendations.

For each recommendation include:
1. Recommendation title
2. Suggested business action
3. Expected business benefit
4. Priority (High, Medium, or Low)

Requirements:
- Base recommendations only on the provided insights.
- Do not invent statistics.
- Do not assume market growth or historical trends.
- Keep recommendations practical and concise.
- Use professional business language.
"""

response = ollama.chat(
    model="qwen2.5:3b",
    messages=[
        {"role": "user", "content": recommendation_prompt}
    ],
    options={"temperature": 0.2}
)

business_recommendations = response["message"]["content"]

print("===== AI BUSINESS RECOMMENDATIONS =====\n")
print(business_recommendations)

===== AI BUSINESS RECOMMENDATIONS =====

### Recommendation 1: Enhance Visibility for Remote Job Listings

**Recommendation Title:** Increase Visibility for Remote Job Listings

**Suggested Business Action:** Develop a targeted marketing campaign to highlight remote job listings in Kerala and other high-demand regions. Utilize social media platforms, job boards, and local business networks to promote these opportunities.

**Expected Business Benefit:** Attract a broader pool of candidates who are looking for remote work opportunities, thereby increasing the number of applications and potentially reducing the time to fill positions.

**Priority:** High

### Recommendation 2: Improve Salary Transparency

**Recommendation Title:** Improve Salary Transparency in Job Listings

**Suggested Business Action:** Implement a policy to provide more detailed salary information in job listings, including the average and median salaries. This will help candidates make more informed decisions and pote

In [10]:
validation_prompt = f"""
Review and revise the following AI-generated content.

Verified data insights:
{chr(10).join(insights)}

Executive summary:
{executive_summary}

Business recommendations:
{business_recommendations}

Rules:
- Remove unsupported claims about market growth.
- Do not assume remote jobs are concentrated in Kerala.
- Do not assume delivery jobs are concentrated in Kerala.
- Keep all verified statistics unchanged.
- Describe expected business benefits as potential outcomes.
- Provide a revised executive summary of approximately 200 words
  and 5 actionable business recommendations.
"""

response = ollama.chat(
    model="qwen2.5:3b",
    messages=[
        {"role": "user", "content": validation_prompt}
    ],
    options={"temperature": 0.1}
)

validated_report = response["message"]["content"]

print("===== REVIEWED BUSINESS REPORT =====\n")
print(validated_report)

===== REVIEWED BUSINESS REPORT =====

### Revised Executive Summary

This summary provides key insights from the Indian Job Market dataset, highlighting current trends and opportunities. The analysis reveals that Kerala leads with 159 job listings, indicating strong demand in this region. The most frequently listed job title is Delivery Partner/Delivery Executive with 9 openings, suggesting a growing need for delivery services. The average monthly salary is INR 22,850.86, with a median of INR 18,500.00, highlighting the competitive nature of the job market. Remote opportunities represent 10.72% of the dataset, representing a significant portion of the market. However, 40.72% of listings lack usable monthly salary information, presenting a challenge for salary benchmarking and negotiation.

These insights underscore the importance of understanding regional market dynamics and the evolving job market landscape. For businesses, the opportunity lies in leveraging the growing demand for del

In [11]:
final_summary = """
The Indian Job Market dataset contains 830 unique job listings
after removing five duplicate records. Kerala accounts for the
highest number of listings, with 159 opportunities, making it
a relevant region for recruitment planning within this dataset.

Delivery Partner/Delivery Executive is the most frequently
listed job title, with nine openings. However, this frequency
alone does not establish a market growth trend.

The corrected average monthly salary is INR 22,850.86,
while the median is INR 18,500. Salary benchmarks are based
on 492 usable salary records.

Remote opportunities account for 89 listings, representing
10.72% of the dataset. Businesses could explore remote
recruitment strategies to reach a wider candidate pool.

Approximately 40.72% of listings lack usable monthly salary
information, limiting the reliability of compensation comparisons.
Improving salary disclosure and standardizing salary periods
could support better recruitment decisions.

Businesses should prioritize regional recruitment planning,
transparent compensation ranges, accurate job descriptions,
and targeted remote recruitment campaigns. These actions
could improve candidate engagement and recruitment efficiency.

The findings represent the available dataset rather than
the entire Indian job market. Additional historical data
and role-specific analysis would be required to establish
market growth trends and evaluate long-term business opportunities.
""".strip()

final_recommendations = [
    "HIGH: Prioritize salary transparency and standardized salary ranges.",
    "HIGH: Evaluate recruitment opportunities in Kerala.",
    "HIGH: Optimize job descriptions for frequently listed roles.",
    "MEDIUM: Promote remote opportunities across suitable regions.",
    "MEDIUM: Improve recruitment data quality and candidate experience."
]

print("===== FINAL VALIDATED SUMMARY =====")
print(final_summary)
print("\nWord count:", len(final_summary.split()))

print("\n===== FINAL RECOMMENDATIONS =====")
for i, rec in enumerate(final_recommendations, 1):
    print(f"{i}. {rec}")

print("\nContent validation completed successfully!")

===== FINAL VALIDATED SUMMARY =====
The Indian Job Market dataset contains 830 unique job listings
after removing five duplicate records. Kerala accounts for the
highest number of listings, with 159 opportunities, making it
a relevant region for recruitment planning within this dataset.

Delivery Partner/Delivery Executive is the most frequently
listed job title, with nine openings. However, this frequency
alone does not establish a market growth trend.

The corrected average monthly salary is INR 22,850.86,
while the median is INR 18,500. Salary benchmarks are based
on 492 usable salary records.

Remote opportunities account for 89 listings, representing
10.72% of the dataset. Businesses could explore remote
recruitment strategies to reach a wider candidate pool.

Approximately 40.72% of listings lack usable monthly salary
information, limiting the reliability of compensation comparisons.
Improving salary disclosure and standardizing salary periods
could support better recruitment dec

In [18]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.chart.data import CategoryChartData
from pptx.enum.chart import XL_CHART_TYPE
from IPython.display import display, FileLink
from pathlib import Path

prs = Presentation()
prs.slide_width = Inches(13.33)
prs.slide_height = Inches(7.5)

# Helper function for text slides
def add_text_slide(title, content, font_size=24):
    slide = prs.slides.add_slide(prs.slide_layouts[1])
    slide.shapes.title.text = title

    tf = slide.placeholders[1].text_frame
    tf.clear()
    tf.word_wrap = True

    for i, item in enumerate(content):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.text = item
        p.font.size = Pt(font_size)
        p.space_after = Pt(12)

    return slide


# Helper function for chart slides
def add_chart_slide(title, insight, categories, values, series_name):

    slide = prs.slides.add_slide(prs.slide_layouts[5])
    slide.shapes.title.text = title

    chart_data = CategoryChartData()
    chart_data.categories = categories
    chart_data.add_series(series_name, values)

    slide.shapes.add_chart(
        XL_CHART_TYPE.COLUMN_CLUSTERED,
        Inches(1),
        Inches(1.4),
        Inches(11.3),
        Inches(4.5),
        chart_data
    )

    textbox = slide.shapes.add_textbox(
        Inches(1),
        Inches(6),
        Inches(11.3),
        Inches(1)
    )

    tf = textbox.text_frame
    tf.word_wrap = True
    tf.text = insight
    tf.paragraphs[0].font.size = Pt(20)

    return slide


# Slide 1: Title
slide = prs.slides.add_slide(prs.slide_layouts[0])

slide.shapes.title.text = "Indian Job Market Analytics"

slide.placeholders[1].text = (
    "AI Executive Summary & Business Recommendations\n"
    "Week 8 – Day 2 | Data Analyst Internship\n"
    f"Analyzed Job Listings: {len(df_clean)}"
)

# Store validated executive summary in speaker notes
slide.notes_slide.notes_text_frame.text = final_summary


# Slide 2: Geographic Demand
top_states = state_counts.head(5)

add_chart_slide(
    "Insight 1 – Geographic Demand",
    insights[0],
    top_states.index.tolist(),
    top_states.tolist(),
    "Job Listings"
)


# Slide 3: Job Demand
top_jobs = job_counts.head(5)

add_chart_slide(
    "Insight 2 – Job Demand",
    insights[1],
    top_jobs.index.tolist(),
    top_jobs.tolist(),
    "Job Listings"
)


# Slide 4: Salary Benchmark
add_chart_slide(
    "Insight 3 – Salary Benchmark",
    insights[2],
    ["Average Salary", "Median Salary"],
    [float(salary.mean()), float(salary.median())],
    "Monthly Salary (INR)"
)


# Slide 5: Remote Opportunities
add_chart_slide(
    "Insight 4 – Remote Opportunities",
    insights[3],
    ["Remote", "Other Listings"],
    [
        int(remote_count),
        int(len(df_clean) - remote_count)
    ],
    "Job Listings"
)


# Slide 6: Salary Transparency
add_chart_slide(
    "Insight 5 – Salary Transparency",
    insights[4],
    ["Usable Salary", "Missing Usable Salary"],
    [
        int(salary.notna().sum()),
        int(salary.isna().sum())
    ],
    "Job Listings"
)


# Slide 7: Business Recommendations
add_text_slide(
    "Actionable Business Recommendations",
    final_recommendations,
    22
)


# Save PowerPoint
ppt_path = Path("W8D2_Business_Recommendations.pptx")

prs.save(str(ppt_path))


# Verify generated file
print("===== POWERPOINT EXPORT =====")
print("PowerPoint generated successfully!")
print("Total slides:", len(prs.slides))
print("File name:", ppt_path.name)
print("File size:", ppt_path.stat().st_size, "bytes")

display(FileLink(str(ppt_path)))

===== POWERPOINT EXPORT =====
PowerPoint generated successfully!
Total slides: 7
File name: W8D2_Business_Recommendations.pptx
File size: 71436 bytes


c:\Users\Prave\OneDrive\Desktop\SQL-Day1\SQL-Day3\Pandas-Advanced\Week-3-Day-5\Week-4\Week-4-Day-5-Capstone\Week-5-Day-1-PandasAI\Week-5-Day-3-DuckDB\Week-6\W6D5-NL-SQL-Dashboard\week_8\W8D2-Business-Recommendations\W8D2_Business_Recommendations.pptx

In [16]:
import os

os.startfile(str(ppt_path.resolve()))

In [19]:
from pptx import Presentation

report = Presentation(str(ppt_path))

assert not df.empty
assert df_clean.duplicated().sum() == 0
assert len(insights) == 5
assert 180 <= len(final_summary.split()) <= 220
assert len(final_recommendations) == 5
assert ppt_path.exists()
assert len(report.slides) == 7

for i in range(1, 6):
    assert any(
        shape.has_chart for shape in report.slides[i].shapes
    ), f"Chart missing in slide {i + 1}"

assert final_summary in report.slides[0].notes_slide.notes_text_frame.text

print("===== FINAL WORKFLOW VALIDATION =====")
print("CSV Loading: PASS")
print("Data Cleaning: PASS")
print("EDA and Insights: PASS")
print("Executive Summary: PASS")
print("Business Recommendations: PASS")
print("7-Slide PPT Export: PASS")
print("5 Individual Insight Charts: PASS")
print("Executive Summary Speaker Notes: PASS")

print("\nAll validation checks passed successfully!")

===== FINAL WORKFLOW VALIDATION =====
CSV Loading: PASS
Data Cleaning: PASS
EDA and Insights: PASS
Executive Summary: PASS
Business Recommendations: PASS
7-Slide PPT Export: PASS
5 Individual Insight Charts: PASS
Executive Summary Speaker Notes: PASS

All validation checks passed successfully!
